EEG Analysis Project: A Study of Auditory Lateralization
Author: Ismaïl-Ayoub EL GOTE
Date: 18 Feb 2026
Course / Context: X Reality and Psychology 2


1. Introduction
Electroencephalography (EEG) allows for the measurement of cerebral electrical activity with millisecond temporal resolution. This project aims to analyze brain responses evoked by auditory stimuli presented binaurally (left ear vs. right ear).

1.1 Scientific Objectives
The primary objective is to highlight the phenomenon of hemispheric lateralization in auditory processing.

1.2 Hypotheses
Time Domain (ERP): We expect to observe a prominent N100 component (negativity around 100ms) over temporal and central electrodes.

Topography (Lateralization): Consistent with the organization of auditory pathways, stimulation to the left ear should elicit a predominant cortical response in the right hemisphere (contralateral), and vice versa.

Time-Frequency Domain: We anticipate an increase in spectral power in lower frequencies (< 15 Hz) synchronized with stimulus onset (evoked and induced activity).

In [ ]:
# Import libraries
import mne
import numpy as np
import matplotlib.pyplot as plt
from mne.preprocessing import ICA

#Set visualization parameters
%matplotlib inline

2. Data Loading and Configuration
We will use the MNE Sample Dataset, which contains multimodal (MEG/EEG) recordings from a passive audiovisual stimulation task.

2.1 Selection Strategy
To simulate a standard clinical acquisition, we will restrict the analysis to:

EEG channels (60 electrodes, standard 10-20 montage).

EOG (Electrooculogram) channels for artifact correction.

MEG channels (Magnetometers and Gradiometers) will be excluded

In [2]:
# --- CONFIGURATION PATHS ---
# Define data paths dynamically using MNE's built-in dataset function
sample_data_folder = mne.datasets.sample.data_path()
sample_data_raw_file = sample_data_folder / 'MEG' / 'sample' / 'sample_audvis_raw.fif'

# --- DATA LOADING ---
# Load raw data with preload=True to allow future processing
print("Loading data...")
raw = mne.io.read_raw_fif(sample_data_raw_file, preload=True, verbose=False)

# --- CHANNEL SELECTION ---
# Keep only EEG, EOG (for artifacts), and Stimulus channels
# We explicitly exclude MEG channels to simulate a standard EEG setup
raw.pick_types(meg=False, eeg=True, eog=True, stim=True)

# --- DISPLAY INFO ---
print("\n--- Data Information ---")
print(raw.info)
print(f"\nDimensions: {len(raw.ch_names)} channels x {raw.n_times} time points")
print(f"Sampling Frequency: {raw.info['sfreq']} Hz")

Using default location ~/mne_data for sample...
Creating /Users/ismail/mne_data


  0%|                                              | 0.00/1.65G [00:00<?, ?B/s]

OSError: [Errno 28] No space left on device

3. Signal Preprocessing
The quality of raw data necessitates rigorous cleaning to isolate brain activity from physiological and environmental noise.

3.1 Frequency Filtering and Re-referencing
We will apply the following filters:

High-pass filter: To eliminate slow drifts (sweat, slow movements).

Low-pass filter: To remove high-frequency noise while preserving the Gamma band for time-frequency analysis.

Notch filter: To suppress power line noise.

Re-referencing: The data will be re-referenced to an Average Reference, a current standard to minimize the bias of a single physical reference electrode.

In [ ]:
--- FILTER PARAMETERS ---

Define cut-off frequencies based on methodology

--- APPLY FILTERS ---

Apply High-pass, Low-pass, and Notch filters to 'raw'

--- RE-REFERENCING ---

Set EEG reference to 'average' with projection=True

3.2 Artifact Correction using ICA (Independent Component Analysis)
ICA is used to statistically separate signal sources.

Methodological Strategy:
As ICA is sensitive to low-frequency drifts, we will adopt a two-pass strategy:

Calculate (Fit) ICA on a copy of the data severely filtered (e.g., 1.0 Hz).

Apply the demixing matrix to the analysis data (e.g., filtered at 0.1 Hz).

This allows for robust detection of eye blinks (EOG) without altering slow cognitive components (such as P300 or CNV).

In [ ]:
--- ICA CONFIGURATION ---

Define ICA parameters (n_components, method='picard' or 'fastica')

--- ICA FITTING ---

1. Create a copy of raw data filtered at e.g. 1.0 Hz for stability

2. Fit ICA on this copy

--- ARTIFACT DETECTION ---

Automatically find components correlating with EOG channel

Plot scores and sources to verify

--- APPLY ICA ---

Exclude bad components and apply to the ORIGINAL 'raw' object

4. Epoch Extraction
We segment the continuous signal around events of interest.

4.1 Condition Definition
We will focus on the auditory contrast:

Condition A: Auditory Stimulus / Left Ear

Condition B: Auditory Stimulus / Right Ear

4.2 Time Windowing
The analysis window is defined broadly (e.g., -0.5s to +1.0s) for two reasons:

To observe the pre-stimulus baseline.

To avoid edge effects during the Wavelet Transform in the next step.

In [ ]:
--- EVENT DETECTION ---

Find events in the stimulus channel

--- EVENT DICTIONARY ---

Map integer codes to descriptive strings (e.g., 'Auditory/Left', 'Auditory/Right')

--- EPOCHING PARAMETERS ---

Define tmin, tmax, baseline correction, and rejection thresholds

--- CREATE EPOCHS ---

Generate the Epochs object

Check drop_log to verify data quality

5. Time-Frequency Analysis (1-50 Hz)
This step aims to quantify the oscillatory energy of the signal between 1 and 50 Hz.

5.1 Methodology: Morlet Wavelets
We use Morlet wavelet convolution to obtain a time-frequency representation.

Adaptive Cycles: The number of cycles will increase with frequency to optimize the trade-off between temporal resolution (low frequencies) and frequency resolution (high frequencies).

Normalization: A spectral baseline correction (Log-Ratio dB) will be applied to highlight relative power changes compared to the pre-stimulus silence.

In [ ]:
--- TFR PARAMETERS ---

Define frequencies (e.g., 1-50Hz) and number of cycles (adaptive)

--- COMPUTE TFR ---

Calculate Morlet Wavelet Transform on 'Auditory' epochs

return_itc=False (focus on Power)

--- VISUALIZATION ---

Plot TFR map for a representative temporal electrode

Apply baseline mode='logratio'

6. Estimation of Evoked Responses (ERP)
The analysis of Event-Related Potentials (ERPs) allows for isolating the "phase-locked" response (perfectly synchronized to the stimulus) through averaging.

We will specifically look for the N100 component (early auditory response).

In [ ]:
--- AVERAGING ---

Compute Evoked objects for 'Auditory/Left' and 'Auditory/Right'

--- VISUALIZATION (Butterfly Plot) ---

Plot the global field power or butterfly plot to identify main peaks

7. Results: Lateralization Analysis
This section forms the core of the scientific demonstration. We will compare the topography of brain responses based on the side of stimulation.

7.1 Topographical Maps (Topomaps)
Visual comparison of the spatial distribution of voltage at the N100 peak (~100ms).

In [ ]:
--- TOPOMAP PARAMETERS ---

Select time point around N100 peak

--- PLOT TOPOMAPS ---

Display Topomaps for Left vs Right stimulation side-by-side

7.2 Region of Interest (ROI) Comparison
To statistically validate the effect, we compare the signal amplitude on symmetrical electrode groups (Left Temporal vs. Right Temporal).

Expected Outcome: The N100 amplitude should be maximal over the ROI contralateral to the stimulus.

In [ ]:
--- ROI DEFINITION ---

Define lists of channels for Left Temporal and Right Temporal ROIs

--- ROI PLOTTING ---

Plot comparison of Evoked responses specifically on these ROIs

8. Discussion and Conclusion
(To be drafted after analysis)

This section will summarize:

The validation (or refutation) of the initial hypotheses regarding the N100 and lateralization.

The interpretation of observed oscillatory dynamics (TFR).

The methodological limitations of the study (e.g., average reference, volume conduction).

Perspectives for future improvements.